In [1]:
from unsloth import FastLanguageModel
import evaluate
import re
import torch
from tqdm import tqdm

# 1. Load the Base Model fresh (without LoRA)
print("Loading Base Model...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/sachithra/miniconda3/envs/unsloth_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Base Model...
==((====))==  Unsloth 2025.12.5: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.628 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [2]:
# Defined the prompt format for Unit Testing
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = base_tokenizer.eos_token

def formatting_prompts_func(examples):
    questions = examples["question"]
    codes     = examples["code_ground_truth"]
    unit_tests_lists = examples["unit_tests"]
    
    texts = []
    # Zip through the batch
    for question, code, tests_list in zip(questions, codes, unit_tests_lists):
        
        # safely extract the test suite
        if tests_list:
            if isinstance(tests_list, dict) and 'code' in tests_list:
                full_test_suite = "\n\n".join([str(c) for c in tests_list['code'] if c])
            elif isinstance(tests_list, list):
                full_test_suite = "\n\n".join([str(t['code']) for t in tests_list if isinstance(t, dict) and t.get('code')])
            elif isinstance(tests_list, str):
                full_test_suite = tests_list
            else:
                full_test_suite = ""
        else:
            full_test_suite = ""
            
        # We combine the Question and the Code into the Input field
        complex_input = f"Problem Description:\n{question}\n\nCode to Test:\n{code}"
        instruction = "Write a comprehensive Python unit test suite for the provided code."
        
        text = alpaca_prompt.format(instruction, complex_input, full_test_suite.strip()) + EOS_TOKEN
        texts.append(text)
        
    return { "text" : texts }

from datasets import load_dataset

# Load both the train and test splits of the dataset
train_dataset = load_dataset("KAKA22/CodeRM-UnitTest", split="train")
test_dataset = load_dataset("KAKA22/CodeRM-UnitTest", split="test")

#Shuffle and take a smaller subset of the test dataset for faster evaluation
test_dataset = test_dataset.shuffle(seed=42).select(range(1000))

# Apply the formatting function to both datasets
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
test_dataset = test_dataset.map(formatting_prompts_func, batched=True)

In [3]:


# Prepare for inference
FastLanguageModel.for_inference(base_model)

rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

predictions = []
references = []

# Use the same evaluation subset
eval_sample_size = 100
eval_subset = test_dataset.select(range(eval_sample_size))

print(f"Generating test cases with BASE MODEL for {eval_sample_size} samples...")
for i in tqdm(range(len(eval_subset))):
    sample = eval_subset[i]
    
    # Format the prompt
    complex_input = f"Problem Description:\n{sample['question']}\n\nCode to Test:\n{sample['code_ground_truth']}"
    instruction = "Write a comprehensive Python unit test suite for the provided code."
    prompt = alpaca_prompt.format(instruction, complex_input, "")
    
    # Tokenize and Generate using base_model
    inputs = base_tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = base_model.generate(**inputs, max_new_tokens=512, use_cache=True, pad_token_id=base_tokenizer.eos_token_id)
    
    decoded_output = base_tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    try:
        response_text = decoded_output.split("### Response:")[1].strip()
        
        # Try to extract python code
        code_blocks = re.findall(r'```(?:python)?\n(.*?)\n```', response_text, re.DOTALL)
        if code_blocks:
            response_text = "\n\n".join(code_blocks).strip()
            
    except IndexError:
        response_text = ""
        
    # Extract ground truth
    try:
        ut = sample['unit_tests']
        if isinstance(ut, dict) and 'code' in ut:
            ground_truth = "\n\n".join([str(c) for c in ut['code'] if c]).strip()
        elif isinstance(ut, list):
            ground_truth = "\n\n".join([str(t['code']) for t in ut if isinstance(t, dict) and t.get('code')]).strip()
        elif isinstance(ut, str):
            ground_truth = ut.strip()
        else:
            ground_truth = ""
    except Exception as e:
        ground_truth = ""
        
    predictions.append(response_text)
    references.append([ground_truth])

# Calculate Scores
print("\n--- Base Model Evaluation Results ---")

filtered_predictions = []
filtered_references = []
empty_preds_count = 0

for pred, ref in zip(predictions, references):
    if not ref[0].strip():
        continue
    if not pred.strip():
        empty_preds_count += 1
        pred = "# empty" 
        
    filtered_predictions.append(pred)
    filtered_references.append(ref)

if empty_preds_count > 0:
    print(f"⚠️ Warning: The base model generated empty responses for {empty_preds_count} samples.")

if len(filtered_references) > 0:
    rouge_results = rouge_metric.compute(
        predictions=filtered_predictions, 
        references=[ref[0] for ref in filtered_references]
    )
    bleu_results = bleu_metric.compute(
        predictions=filtered_predictions, 
        references=filtered_references
    )

    print(f"Base BLEU Score:   {bleu_results['bleu'] * 100:.2f}%")
    print(f"Base ROUGE-1:      {rouge_results['rouge1'] * 100:.2f}%")
    print(f"Base ROUGE-2:      {rouge_results['rouge2'] * 100:.2f}%")
    print(f"Base ROUGE-L:      {rouge_results['rougeL'] * 100:.2f}%")

Generating test cases with BASE MODEL for 100 samples...


100%|██████████| 100/100 [10:09<00:00,  6.10s/it]



--- Base Model Evaluation Results ---
⚠️ Warning: The base model generated empty responses for 16 samples.
Base BLEU Score:   0.00%
Base ROUGE-1:      0.38%
Base ROUGE-2:      0.21%
Base ROUGE-L:      0.34%
